<a href="https://colab.research.google.com/github/TAUforPython/Graph-MachineLearning/blob/main/utils_ERD2MMD_graph_visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ERD to Python Graph Visualization
# Install required libraries
!pip install -q networkx matplotlib pandas pillow
!apt-get install -y graphviz graphviz-dev
!pip install -q pygraphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libgraphviz-dev' instead of 'graphviz-dev'
graphviz is already the newest version (2.42.2-6ubuntu0.1).
The following additional packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk librsvg2-common
  libxcomposite1 libxdot4
Suggested packages:
  gvfs
The following NEW packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgraphviz-dev
  libgtk2.0-0 libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk
  librsvg2-common libxcomposite1 libxdot4
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,496 kB of archives.
After this operation, 7,963 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main a

In [2]:


import re
import xml.etree.ElementTree as ET
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Image as IPImage
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
from PIL import Image
import io
import base64

In [3]:


# @title Parse ERD File
def parse_erd_file(file_content):
    """Parse the ERD XML file and extract entities and relationships"""
    try:
        # Parse XML
        root = ET.fromstring(file_content)

        # Extract entities
        entities = {}
        relations_list = []

        data_source = root.find('.//data-source')
        if data_source is not None:
            for entity in data_source.findall('entity'):
                entity_id = entity.get('id')
                entity_name = entity.get('name')
                if entity_name:
                    # Clean up entity name (remove schema prefix if present)
                    if '.' in entity_name:
                        entity_name = entity_name.split('.')[-1]
                    entities[entity_id] = {
                        'name': entity_name,
                        'x': float(entity.get('x', 0)),
                        'y': float(entity.get('y', 0))
                    }

        # Extract relationships
        relations_elem = root.find('relations')
        if relations_elem is not None:
            for relation in relations_elem.findall('relation'):
                rel_name = relation.get('name', '')
                rel_type = relation.get('type', '')
                pk_ref = relation.get('pk-ref', '')
                fk_ref = relation.get('fk-ref', '')

                if pk_ref in entities and fk_ref in entities:
                    relations_list.append({
                        'name': rel_name,
                        'type': rel_type,
                        'source': entities[pk_ref]['name'],
                        'target': entities[fk_ref]['name'],
                        'source_id': pk_ref,
                        'target_id': fk_ref
                    })

        return entities, relations_list

    except ET.ParseError as e:
        print(f"Error parsing XML: {e}")
        return None, None

# @title Create NetworkX Graph
def create_relationship_graph(entities, relations, max_nodes=30):
    """Create a NetworkX graph from entities and relationships"""
    G = nx.DiGraph()

    # Count relationship frequency
    rel_counter = {}
    for rel in relations:
        key = (rel['source'], rel['target'])
        rel_counter[key] = rel_counter.get(key, 0) + 1

    # Add nodes (entities)
    node_sizes = {}
    for rel in relations[:max_nodes * 2]:
        source = rel['source']
        target = rel['target']

        # Count relationships per entity
        node_sizes[source] = node_sizes.get(source, 0) + 1
        node_sizes[target] = node_sizes.get(target, 0) + 1

        if source not in G.nodes:
            G.add_node(source)
        if target not in G.nodes:
            G.add_node(target)

    # Add edges (relationships)
    for (source, target), count in rel_counter.items():
        if source in G.nodes and target in G.nodes:
            G.add_edge(source, target, weight=count, label=f"{count} rel")

    return G, node_sizes

# @title Hierarchical Graph Layout
def create_hierarchical_layout(G):
    """Create a hierarchical layout for better visualization"""
    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except:
        # Fallback to spring layout if graphviz not available
        pos = nx.spring_layout(G, k=3, iterations=50)
    return pos

# @title Visualize with Matplotlib
def visualize_erd_matplotlib(G, node_sizes, figsize=(20, 12), title="Database ERD Visualization"):
    """Visualize the ERD using matplotlib"""

    fig, ax = plt.subplots(figsize=figsize)

    # Create layout
    try:
        pos = create_hierarchical_layout(G)
    except:
        pos = nx.spring_layout(G, k=3, iterations=50, seed=42)

    # Calculate node sizes based on degree
    node_size_list = []
    for node in G.nodes():
        size = node_sizes.get(node, 1) * 800 + 1000
        node_size_list.append(min(size, 5000))  # Cap maximum size

    # Draw nodes
    nodes = nx.draw_networkx_nodes(
        G, pos,
        node_size=node_size_list,
        node_color='lightblue',
        edgecolors='darkblue',
        linewidths=2,
        alpha=0.9
    )

    # Draw edges with varying thickness based on weight
    edges = G.edges(data=True)
    edge_weights = [d['weight'] * 2 for (u, v, d) in edges]

    nx.draw_networkx_edges(
        G, pos,
        width=edge_weights,
        alpha=0.5,
        edge_color='gray',
        arrows=True,
        arrowsize=20,
        arrowstyle='->',
        connectionstyle='arc3,rad=0.1'
    )

    # Draw labels
    nx.draw_networkx_labels(
        G, pos,
        font_size=10,
        font_weight='bold',
        font_family='sans-serif'
    )

    # Add edge labels for important relationships
    edge_labels = {(u, v): f"{d['weight']}" for u, v, d in edges if d['weight'] > 1}
    if edge_labels:
        nx.draw_networkx_edge_labels(
            G, pos,
            edge_labels=edge_labels,
            font_size=8,
            font_color='red'
        )

    # Customize plot
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')

    # Add legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue',
                   markersize=15, label='Entity (size = relationships)'),
        plt.Line2D([0], [0], color='gray', lw=2, label='Relationship'),
        plt.Line2D([0], [0], color='gray', lw=4, alpha=0.5, label='Multiple relationships')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1))

    plt.tight_layout()
    return fig

# @title Circular Layout Visualization
def visualize_circular_erd(G, node_sizes, figsize=(16, 16)):
    """Visualize ERD in circular layout for large graphs"""

    fig, ax = plt.subplots(figsize=figsize)

    # Use circular layout
    pos = nx.circular_layout(G)

    # Draw nodes
    node_size_list = [min(node_sizes.get(node, 1) * 500 + 800, 3000) for node in G.nodes()]

    nx.draw_networkx_nodes(
        G, pos,
        node_size=node_size_list,
        node_color='lightgreen',
        edgecolors='darkgreen',
        linewidths=2,
        alpha=0.9
    )

    # Draw edges
    nx.draw_networkx_edges(
        G, pos,
        width=1,
        alpha=0.3,
        edge_color='gray',
        arrows=True,
        arrowsize=15
    )

    # Draw labels with rotation
    for node, (x, y) in pos.items():
        angle = np.arctan2(y, x) * 180 / np.pi
        if angle < 0:
            angle += 360
        ha = 'left' if -90 <= angle <= 90 else 'right'

        ax.annotate(
            node, xy=(x, y),
            fontsize=8,
            fontweight='bold',
            ha=ha,
            va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
        )

    ax.set_title("Circular ERD View - Entity Relationships", fontsize=16, fontweight='bold')
    ax.axis('equal')
    ax.axis('off')

    return fig

# @title Interactive Bokeh Visualization (if bokeh is available)
def try_bokeh_visualization(G, node_sizes):
    """Attempt to create interactive visualization with bokeh"""
    try:
        from bokeh.io import output_notebook, show
        from bokeh.models import Plot, Range1d, MultiLine, Circle, HoverTool, TapTool, BoxSelectTool
        from bokeh.models.graphs import from_networkx, NodesAndLinkedEdges
        from bokeh.palettes import Spectral4
        from bokeh.plotting import figure

        output_notebook()

        # Create plot
        plot = figure(
            title="Interactive ERD Graph",
            x_range=Range1d(-1.1, 1.1),
            y_range=Range1d(-1.1, 1.1),
            tools="pan,wheel_zoom,save,reset",
            active_scroll='wheel_zoom',
            sizing_mode='stretch_width',
            height=600
        )

        # Create graph renderer
        graph_renderer = from_networkx(G, nx.spring_layout, scale=2, center=(0, 0))

        # Set node properties
        graph_renderer.node_renderer.glyph = Circle(
            size={'field': 'size', 'default': 20},
            fill_color='lightblue',
            fill_alpha=0.8
        )

        # Set edge properties
        graph_renderer.edge_renderer.glyph = MultiLine(
            line_color='#CCCCCC',
            line_alpha=0.8,
            line_width={'field': 'weight', 'default': 1}
        )

        # Add hover tool
        hover_tool = HoverTool(
            tooltips=[("Entity", "@index"), ("Relationships", "@size")]
        )
        plot.add_tools(hover_tool, TapTool(), BoxSelectTool())

        # Set node attributes
        sizes = [node_sizes.get(node, 1) * 5 + 10 for node in G.nodes()]
        graph_renderer.node_renderer.data_source.data['size'] = sizes

        plot.renderers.append(graph_renderer)

        return plot

    except ImportError:
        print("Bokeh not available. Install with: pip install bokeh")
        return None

# @title Create Static HTML with D3.js
def create_d3_html(G, node_sizes):
    """Create an HTML file with D3.js visualization"""

    # Convert graph to JSON
    nodes = [{"id": node, "size": node_sizes.get(node, 1)} for node in G.nodes()]
    links = [{"source": u, "target": v, "weight": d.get('weight', 1)}
             for u, v, d in G.edges(data=True)]

    graph_data = {"nodes": nodes, "links": links}

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>ERD Visualization</title>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body {{ margin: 0; font-family: Arial, sans-serif; }}
            #controls {{
                position: fixed;
                top: 10px;
                right: 10px;
                background: white;
                padding: 15px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.2);
                z-index: 1000;
            }}
            .control-btn {{
                display: block;
                width: 100%;
                margin: 5px 0;
                padding: 8px;
                background: #4CAF50;
                color: white;
                border: none;
                border-radius: 4px;
                cursor: pointer;
            }}
            .control-btn:hover {{ background: #45a049; }}
            .node {{
                stroke: #fff;
                stroke-width: 2px;
                cursor: pointer;
            }}
            .node:hover {{
                stroke: #333;
                stroke-width: 3px;
            }}
            .link {{
                stroke: #999;
                stroke-opacity: 0.6;
            }}
            .node-label {{
                font-size: 12px;
                pointer-events: none;
                text-shadow: 0 1px 0 #fff;
            }}
            #tooltip {{
                position: absolute;
                background: rgba(0,0,0,0.8);
                color: white;
                padding: 5px 10px;
                border-radius: 4px;
                font-size: 12px;
                pointer-events: none;
                opacity: 0;
                transition: opacity 0.2s;
            }}
        </style>
    </head>
    <body>
        <div id="controls">
            <h3>Controls</h3>
            <button class="control-btn" onclick="zoomIn()">Zoom In +</button>
            <button class="control-btn" onclick="zoomOut()">Zoom Out -</button>
            <button class="control-btn" onclick="resetView()">Reset View</button>
            <button class="control-btn" onclick="exportSVG()">Export SVG</button>
            <hr>
            <div id="stats"></div>
        </div>

        <div id="tooltip"></div>

        <svg width="100%" height="100vh"></svg>

        <script>
            const graphData = {json.dumps(graph_data)};

            const width = window.innerWidth;
            const height = window.innerHeight;

            const svg = d3.select("svg");
            const container = svg.append("g");

            // Add zoom behavior
            const zoom = d3.zoom()
                .scaleExtent([0.1, 4])
                .on("zoom", (event) => {{
                    container.attr("transform", event.transform);
                }});

            svg.call(zoom);

            // Create force simulation
            const simulation = d3.forceSimulation(graphData.nodes)
                .force("link", d3.forceLink(graphData.links).id(d => d.id).distance(200))
                .force("charge", d3.forceManyBody().strength(-500))
                .force("center", d3.forceCenter(width / 2, height / 2))
                .force("collision", d3.forceCollide().radius(d => Math.sqrt(d.size) * 10));

            // Create links
            const link = container.append("g")
                .selectAll("line")
                .data(graphData.links)
                .enter().append("line")
                .attr("class", "link")
                .attr("stroke-width", d => Math.sqrt(d.weight));

            // Create nodes
            const node = container.append("g")
                .selectAll("circle")
                .data(graphData.nodes)
                .enter().append("circle")
                .attr("class", "node")
                .attr("r", d => Math.sqrt(d.size) * 10)
                .attr("fill", d => d3.interpolateViridis(d.size / 10))
                .call(d3.drag()
                    .on("start", dragstarted)
                    .on("drag", dragged)
                    .on("end", dragended))
                .on("mouseover", (event, d) => {{
                    d3.select("#tooltip")
                        .style("opacity", 1)
                        .style("left", (event.pageX + 10) + "px")
                        .style("top", (event.pageY - 20) + "px")
                        .html(`<strong>${{d.id}}</strong><br>Relationships: ${{d.size}}`);
                }})
                .on("mouseout", () => {{
                    d3.select("#tooltip").style("opacity", 0);
                }});

            // Add labels
            const label = container.append("g")
                .selectAll("text")
                .data(graphData.nodes)
                .enter().append("text")
                .attr("class", "node-label")
                .attr("dx", d => Math.sqrt(d.size) * 10 + 5)
                .attr("dy", 4)
                .text(d => d.id.length > 20 ? d.id.substring(0, 17) + "..." : d.id);

            // Update positions on simulation tick
            simulation.on("tick", () => {{
                link
                    .attr("x1", d => d.source.x)
                    .attr("y1", d => d.source.y)
                    .attr("x2", d => d.target.x)
                    .attr("y2", d => d.target.y);

                node
                    .attr("cx", d => d.x)
                    .attr("cy", d => d.y);

                label
                    .attr("x", d => d.x)
                    .attr("y", d => d.y);
            }});

            // Drag functions
            function dragstarted(event, d) {{
                if (!event.active) simulation.alphaTarget(0.3).restart();
                d.fx = d.x;
                d.fy = d.y;
            }}

            function dragged(event, d) {{
                d.fx = event.x;
                d.fy = event.y;
            }}

            function dragended(event, d) {{
                if (!event.active) simulation.alphaTarget(0);
                d.fx = null;
                d.fy = null;
            }}

            // Control functions
            function zoomIn() {{
                svg.transition().call(zoom.scaleBy, 1.3);
            }}

            function zoomOut() {{
                svg.transition().call(zoom.scaleBy, 0.7);
            }}

            function resetView() {{
                svg.transition().call(zoom.transform, d3.zoomIdentity);
            }}

            function exportSVG() {{
                const svgElement = document.querySelector('svg');
                const serializer = new XMLSerializer();
                let source = serializer.serializeToString(svgElement);
                source = '<?xml version="1.0" encoding="UTF-8"?>' + source;

                const blob = new Blob([source], {{type: 'image/svg+xml'}});
                const url = URL.createObjectURL(blob);
                const link = document.createElement('a');
                link.href = url;
                link.download = 'erd_visualization.svg';
                link.click();
                URL.revokeObjectURL(url);
            }}

            // Update stats
            function updateStats() {{
                const stats = `Nodes: ${{graphData.nodes.length}}<br>
                              Edges: ${{graphData.links.length}}`;
                document.getElementById('stats').innerHTML = stats;
            }}
            updateStats();
        </script>
    </body>
    </html>
    """

    return html_content

# @title Main Visualization Interface
class ERDVisualizer:
    def __init__(self):
        self.G = None
        self.node_sizes = None
        self.entities = None
        self.relations = None

    def load_from_file(self, file_content):
        """Load ERD from file content"""
        self.entities, self.relations = parse_erd_file(file_content)
        if self.entities and self.relations:
            self.G, self.node_sizes = create_relationship_graph(self.entities, self.relations)
            return True
        return False

    def show_matplotlib(self, layout='hierarchical'):
        """Show matplotlib visualization"""
        if self.G is None:
            print("No graph loaded")
            return

        if layout == 'hierarchical':
            fig = visualize_erd_matplotlib(self.G, self.node_sizes)
        else:
            fig = visualize_circular_erd(self.G, self.node_sizes)

        plt.show()
        return fig

    def show_d3(self):
        """Show D3.js visualization"""
        if self.G is None:
            print("No graph loaded")
            return

        html_content = create_d3_html(self.G, self.node_sizes)
        display(HTML(html_content))

    def show_bokeh(self):
        """Show bokeh visualization if available"""
        if self.G is None:
            print("No graph loaded")
            return

        plot = try_bokeh_visualization(self.G, self.node_sizes)
        if plot:
            from bokeh.io import show
            show(plot)

    def print_stats(self):
        """Print graph statistics"""
        if self.G is None:
            print("No graph loaded")
            return

        print("📊 ERD Statistics")
        print("=" * 50)
        print(f"Total Entities: {self.G.number_of_nodes()}")
        print(f"Total Relationships: {self.G.number_of_edges()}")

        # Find most connected entities
        degrees = dict(self.G.degree())
        top_entities = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]

        print("\n🔗 Most Connected Entities:")
        for entity, degree in top_entities:
            print(f"  • {entity}: {degree} relationships")

# @title Upload and Visualize Interface
visualizer = ERDVisualizer()

# Create UI elements
upload_button = widgets.FileUpload(
    accept='.txt, .xml',
    multiple=False,
    description='📁 Upload ERD File'
)

layout_select = widgets.Dropdown(
    options=['hierarchical', 'circular'],
    value='hierarchical',
    description='Layout:'
)

viz_type = widgets.RadioButtons(
    options=['Matplotlib', 'Interactive D3', 'Both'],
    value='Both',
    description='Visualization:'
)

max_nodes = widgets.IntSlider(
    value=40,
    min=10,
    max=100,
    step=5,
    description='Max Entities:'
)

load_button = widgets.Button(
    description='📊 Load and Visualize',
    button_style='primary'
)

output = widgets.Output()
stats_output = widgets.Output()

def on_load_clicked(b):
    with output:
        clear_output(wait=True)

        if not upload_button.value:
            print("Please upload a file first.")
            return

        # Get file content
        uploaded_file = list(upload_button.value.values())[0]
        file_content = uploaded_file['content'].decode('utf-8')

        print("🔄 Parsing ERD file...")
        if visualizer.load_from_file(file_content):
            print(f"✅ Loaded {visualizer.G.number_of_nodes()} entities and {visualizer.G.number_of_edges()} relationships")

            # Show statistics
            with stats_output:
                clear_output(wait=True)
                visualizer.print_stats()

            # Generate visualization
            print("\n🎨 Generating visualization...")

            if viz_type.value in ['Matplotlib', 'Both']:
                visualizer.show_matplotlib(layout=layout_select.value)

            if viz_type.value in ['Interactive D3', 'Both']:
                visualizer.show_d3()
        else:
            print("❌ Failed to parse ERD file")

load_button.on_click(on_load_clicked)

# Display UI
print("=" * 60)
print("🎯 DATABASE ERD VISUALIZER")
print("=" * 60)

display(widgets.VBox([
    upload_button,
    widgets.HBox([max_nodes, layout_select]),
    viz_type,
    load_button,
    stats_output,
    output
]))

# @title Export Visualization
def export_current_viz():
    """Export current visualization"""
    if visualizer.G is None:
        print("No visualization loaded")
        return

    # Create high-res matplotlib figure
    fig = visualize_erd_matplotlib(visualizer.G, visualizer.node_sizes, figsize=(30, 20))

    # Save to file
    filename = "erd_visualization.png"
    fig.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')

    print(f"✅ Saved high-resolution image as: {filename}")

    # Offer download
    files.download(filename)

    plt.close(fig)

export_button = widgets.Button(
    description='💾 Export High-Res PNG',
    button_style='success'
)

export_button.on_click(lambda b: export_current_viz())

display(export_button)

print("\n📌 Instructions:")
print("1. Upload your ERD file (the DB_MIS_REPORT.txt file)")
print("2. Adjust max entities to control diagram complexity")
print("3. Choose visualization type:")
print("   • Matplotlib - Static publication-quality images")
print("   • Interactive D3 - Dynamic web-based visualization")
print("   • Both - Show both visualizations")
print("4. Click 'Load and Visualize' to generate the diagram")
print("5. Use export button to save high-resolution images")

🎯 DATABASE ERD VISUALIZER


Button(button_style='success', description='💾 Export High-Res PNG', style=ButtonStyle())


📌 Instructions:
1. Upload your ERD file (the DB_MIS_REPORT.txt file)
2. Adjust max entities to control diagram complexity
3. Choose visualization type:
   • Matplotlib - Static publication-quality images
   • Interactive D3 - Dynamic web-based visualization
   • Both - Show both visualizations
4. Click 'Load and Visualize' to generate the diagram
5. Use export button to save high-resolution images
